# Enhanced Grad-CAM for six-emotion FER — full run on Colab
Runtime → Change runtime type → **GPU (T4)**. Run the cells in order. Results are saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fer_msf_gradcam_runs'
!mkdir -p $OUT

In [ ]:
!git clone https://github.com/Nadi-989/fer-enhanced-gradcam.git
%cd fer-enhanced-gradcam
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name --format=csv

## Data
Upload `fer2013.csv` (or the Kaggle folder zip) and `ckextended.csv` to `MyDrive/fer_data/`.

In [ ]:
DATA = '/content/drive/MyDrive/fer_data'
# FER2013: CSV release
!python scripts/prepare_csv.py --csv $DATA/fer2013.csv --out data/fer2013
# or Kaggle folder release: !unzip -q $DATA/fer2013.zip -d data/fer2013
!python scripts/prepare_csv.py --csv $DATA/ckextended.csv --out data/ckplus48 --pool

## FER2013 — three backbones

In [ ]:
for arch in ['resnet18', 'vgg16', 'efficientnet_b0']:
    !python train.py --config configs/fer2013.yaml --arch {arch}
    !python evaluate_cam.py --run runs/fer2013_{arch}
!python sanity_check.py --run runs/fer2013_resnet18
!python make_figures.py --run runs/fer2013_resnet18 --qualitative --regions --bars

## CK+ — 5-fold cross-validation

In [ ]:
for arch in ['resnet18', 'vgg16', 'efficientnet_b0']:
    for f in range(5):
        !python train.py --config configs/ckplus48.yaml --arch {arch} --fold {f} --out_dir runs/ckplus48_{arch}/fold{f}
        !python evaluate_cam.py --run runs/ckplus48_{arch}/fold{f}
    !python aggregate_cv.py --runs runs/ckplus48_{arch}/fold*

## Tables for the paper

In [ ]:
!cat runs/fer2013_resnet18/cam_summary.md
!cat runs/ckplus48_resnet18/cv_results.json | head -20
!cp -r runs $OUT/